In [2]:
#!pip install -q kagglehub

import kagglehub
path = kagglehub.dataset_download("akshayksingh/kidney-disease-dataset")

print("Đường dẫn lưu dataset:", path)

/home/john-vx/year-3-autumn-term/machine_learning/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Đường dẫn lưu dataset: /home/john-vx/.cache/kagglehub/datasets/akshayksingh/kidney-disease-dataset/versions/1


# Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

import warnings
warnings.filterwarnings('ignore')
plt.style.use('fivethirtyeight')
sns.set()
plt.style.use('ggplot')
%matplotlib inline

# Read Dataset And Information

In [4]:
import os

datasets = os.path.join(path, "kidney_disease.csv")
print(f"{datasets}")

/home/john-vx/.cache/kagglehub/datasets/akshayksingh/kidney-disease-dataset/versions/1/kidney_disease.csv


# Data Preprocessing

In [5]:
# Load dataset
df = pd.read_csv(datasets)

# 1. Loại bỏ các cột không cần thiết
df.drop('id', axis=1, inplace=True, errors='ignore')

# 2. Sửa lỗi định dạng kiểu dữ liệu số
for col in ['pcv', 'wc', 'rc']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Làm sạch dữ liệu văn bản (chuẩn hóa nhãn trước khi split)
def clean_text(df):
    target_cols = df.select_dtypes(include='object').columns
    for col in target_cols:
        df[col] = df[col].str.replace('\t', '').str.strip()
        df[col] = df[col].replace({
            'ckd\t': 'ckd', 'notckd': 'not ckd', 'yes\t': 'yes',
            '\tno': 'no', '\tyes': 'yes', ' yes': 'yes'
        })
    return df

df = clean_text(df)
display(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             391 non-null    float64
 1   bp              388 non-null    float64
 2   sg              353 non-null    float64
 3   al              354 non-null    float64
 4   su              351 non-null    float64
 5   rbc             248 non-null    str    
 6   pc              335 non-null    str    
 7   pcc             396 non-null    str    
 8   ba              396 non-null    str    
 9   bgr             356 non-null    float64
 10  bu              381 non-null    float64
 11  sc              383 non-null    float64
 12  sod             313 non-null    float64
 13  pot             312 non-null    float64
 14  hemo            348 non-null    float64
 15  pcv             329 non-null    float64
 16  wc              294 non-null    float64
 17  rc              269 non-null    float64
 18  h

None

In [6]:
from sklearn.model_selection import train_test_split

# 4. Chia tách dữ liệu (Split-First Policy)
# Phải chia trước khi Imputation/Encoding để tránh Data Leakage
X = df.drop('classification', axis=1)
y = df['classification']

# Sử dụng stratify để giữ nguyên tỷ lệ nhãn, RANDOM_STATE=42 theo Protocol 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Kích thước tập huấn luyện: {X_train.shape}")
print(f"Kích thước tập kiểm tra: {X_test.shape}")

Kích thước tập huấn luyện: (320, 24)
Kích thước tập kiểm tra: (80, 24)


In [14]:
# 5. Xử lý giá trị thiếu (Handling Missing Values) triệt để
num_cols = X_train.select_dtypes(exclude='object').columns
cat_cols = X_train.select_dtypes(include='object').columns

# Điền giá trị thiếu cho cột số
for col in num_cols:
    # Tính toán median từ tập Train
    train_median = X_train[col].median()
    # Nếu toàn bộ cột là NaN, điền bằng 0
    if pd.isna(train_median):
        train_median = 0
    X_train[col] = X_train[col].fillna(train_median)
    X_test[col] = X_test[col].fillna(train_median)

# Điền giá trị thiếu cho cột phân loại
for col in cat_cols:
    # Lấy mode từ tập Train
    mode_series = X_train[col].mode()
    train_mode = mode_series[0] if not mode_series.empty else 'unknown'
    X_train[col] = X_train[col].fillna(train_mode)
    X_test[col] = X_test[col].fillna(train_mode)

print(f"Số lượng null còn lại trong X_train: {X_train.isna().sum().sum()}")
print(f"Số lượng null còn lại trong X_test: {X_test.isna().sum().sum()}")

Số lượng null còn lại trong X_train: 0
Số lượng null còn lại trong X_test: 0


In [8]:
from sklearn.preprocessing import LabelEncoder

# 6. Mã hóa dữ liệu (Feature Encoding) dựa trên danh mục của tập Train
le = LabelEncoder()
y_le = LabelEncoder()

# Mã hóa nhãn mục tiêu
y_train = y_le.fit_transform(y_train)
y_test = y_le.transform(y_test)

# Mã hóa các cột đặc trưng
for col in cat_cols:
    le = LabelEncoder()
    X_train[col] = le.fit_transform(X_train[col])
    # Áp dụng mapping từ Train sang Test, xử lý nhãn lạ nếu có
    X_test[col] = X_test[col].map(lambda s: le.transform([s])[0] if s in le.classes_ else -1)

print("Dữ liệu Train sau khi mã hóa:")
display(X_train.head())

Dữ liệu Train sau khi mã hóa:


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane
380,59.0,60.0,1.020,0.0,0.0,1,1,0,0,113.0,...,15.3,54.0,6500.0,4.9,0,0,0,0,0,0
56,76.0,70.0,1.015,3.0,4.0,1,0,1,0,NaN,...,10.2,30.0,11300.0,3.4,1,1,1,1,1,0
126,70.0,90.0,1.015,0.0,0.0,2,1,0,0,144.0,...,12.0,37.0,8200.0,4.5,1,1,0,1,1,0
371,28.0,60.0,1.025,0.0,0.0,1,1,0,0,79.0,...,17.6,51.0,6500.0,5.0,0,0,0,0,0,0
333,23.0,80.0,1.020,0.0,0.0,1,1,0,0,99.0,...,17.7,46.0,4300.0,5.5,0,0,0,0,0,0


### Thiết lập hệ thống Logging chuẩn (Unified Logger)
Hàm `save_experiment_report` tự động phân loại và lưu trữ kết quả dưới dạng JSON.

In [17]:
import json
import datetime
import os

def save_experiment_report(model_name, category, params, metrics, mechanics=None):
    """
    Lưu trữ báo cáo thí nghiệm dưới dạng JSON.
    category: 'iterative' hoặc 'non-iterative'
    """
    report_path = f"model/{model_name}"
    os.makedirs(report_path, exist_ok=True)
    
    report = {
        "meta": {
            "timestamp": datetime.datetime.now().isoformat(),
            "model_name": model_name,
            "category": category,
            "dataset_version": "Kaggle Kidney v1",
            "random_state": 42,
            "test_size": 0.2
        },
        "params": params,
        "mechanics": mechanics or {},
        "metrics": metrics
    }
    
    file_name = f"{report_path}/experiment_log.json"
    with open(file_name, 'w') as f:
        json.dump(report, f, indent=4)
    
    print(f"[Logger] Đã lưu báo cáo JSON cho {model_name} tại {file_name}")
    return report

# Model

## 1. Logistic Regression 

In [18]:
import os
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, log_loss

# 1. Cấu hình
model_name = "LogisticRegression"
params = {"solver": "liblinear", "C": 1.0, "max_iter": 100}
base_path = f"model/{model_name}"
os.makedirs(base_path, exist_ok=True)

# 2. Huấn luyện (Cơ chế Non-Iterative/Solver-based)
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# 3. Thu thập solver metadata
mechanics = {
    "category": "non-iterative",
    "n_iter_": int(lr.n_iter_[0]),
    "train_loss": float(log_loss(y_train, lr.predict_proba(X_train)))
}

# 4. Đánh giá
y_pred = lr.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average='weighted')
}

# 5. Lưu log JSON (Đã tích hợp lại hàm logger)
save_experiment_report(model_name, "non-iterative", params, metrics, mechanics)

# 6. Lưu trọng số và in báo cáo văn bản
joblib.dump(lr, os.path.join(base_path, "best_weights.pkl"))
print(f"--- Kết quả {model_name} ---\n", classification_report(y_test, y_pred))

[Logger] Đã lưu báo cáo JSON cho LogisticRegression tại model/LogisticRegression/experiment_log.json
--- Kết quả LogisticRegression ---
               precision    recall  f1-score   support

           0       0.94      0.96      0.95        50
           1       0.93      0.90      0.92        30

    accuracy                           0.94        80
   macro avg       0.94      0.93      0.93        80
weighted avg       0.94      0.94      0.94        80



## 2. Decision Tree

In [19]:
from sklearn.tree import DecisionTreeClassifier

# 1. Cấu hình
model_name = "DecisionTree"
params = {"criterion": "entropy", "max_depth": 5, "random_state": 42}

# 2. Huấn luyện (Structural Protocol)
dt = DecisionTreeClassifier(**params)
dt.fit(X_train, y_train)

# 3. Thu thập structural metadata
mechanics = {
    "category": "non-iterative",
    "tree_depth": int(dt.get_depth()),
    "n_leaves": int(dt.get_n_leaves()),
    "feature_importances": dt.feature_importances_.tolist()
}

# 4. Đánh giá
y_pred = dt.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_weighted": precision_score(y_test, y_pred, average='weighted'),
    "recall_weighted": recall_score(y_test, y_pred, average='weighted'),
    "f1_weighted": f1_score(y_test, y_pred, average='weighted')
}

# 5. Lưu log JSON
save_experiment_report(model_name, "non-iterative", params, metrics, mechanics)

# 6. Lưu và in kết quả
joblib.dump(dt, f"model/{model_name}/best_weights.pkl")
print(f"\n--- Kết quả {model_name} ---")
print(classification_report(y_test, y_pred))

[Logger] Đã lưu báo cáo JSON cho DecisionTree tại model/DecisionTree/experiment_log.json

--- Kết quả DecisionTree ---
              precision    recall  f1-score   support

           0       0.98      1.00      0.99        50
           1       1.00      0.97      0.98        30

    accuracy                           0.99        80
   macro avg       0.99      0.98      0.99        80
weighted avg       0.99      0.99      0.99        80



## 3. Support Vector Machine

In [20]:
from sklearn.svm import SVC

# 1. Cấu hình
model_name = "SVM"
params = {"kernel": "linear", "C": 1.0, "probability": True}

# 2. Huấn luyện (Non-Iterative)
svm = SVC(**params)
svm.fit(X_train, y_train)

# 3. Thu thập metadata đặc thù SVM
mechanics = {
    "category": "non-iterative",
    "n_support_": svm.n_support_.tolist(),
    "dual_coef_": "stored_in_weights",
    "convergence": "QP_solver_finished"
}

# 4. Đánh giá
y_pred = svm.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average='weighted')
}

# 5. Lưu log & model
save_experiment_report(model_name, "non-iterative", params, metrics, mechanics)
joblib.dump(svm, f"model/{model_name}/best_weights.pkl")
print(f"--- Kết quả {model_name} ---\n", classification_report(y_test, y_pred))

[Logger] Đã lưu báo cáo JSON cho SVM tại model/SVM/experiment_log.json
--- Kết quả SVM ---
               precision    recall  f1-score   support

           0       1.00      0.96      0.98        50
           1       0.94      1.00      0.97        30

    accuracy                           0.97        80
   macro avg       0.97      0.98      0.97        80
weighted avg       0.98      0.97      0.98        80



## 4. K-Nearest Neighbors - KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# 1. Cấu hình
model_name = "KNN"
params = {"n_neighbors": 5, "metric": "minkowski", "p": 2}

# 2. Huấn luyện (Lazy Learner)
knn = KNeighborsClassifier(**params)
knn.fit(X_train, y_train)

# 3. Thu thập metadata
mechanics = {
    "category": "non-iterative",
    "n_samples_fit_": int(knn.n_samples_fit_),
    "effective_metric_": knn.effective_metric_
}

# 4. Đánh giá
y_pred = knn.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average='weighted')
}

# 5. Lưu log
save_experiment_report(model_name, "non-iterative", params, metrics, mechanics)
joblib.dump(knn, f"model/{model_name}/best_weights.pkl")
print(f"--- Kết quả {model_name} ---\n", classification_report(y_test, y_pred))

[Logger] Đã lưu báo cáo JSON cho KNN tại /model/KNN/experiment_log.json
--- Kết quả KNN ---
               precision    recall  f1-score   support

           0       0.89      0.60      0.71        52
           1       0.53      0.86      0.66        28

    accuracy                           0.69        80
   macro avg       0.71      0.73      0.69        80
weighted avg       0.76      0.69      0.69        80



## 5. XGBoost

In [ ]:
from xgboost import XGBClassifier

# 1. Cấu hình (Di chuyển early_stopping_rounds vào constructor)
model_name = "XGBoost"
params = {
    "n_estimators": 500,
    "learning_rate": 0.1,
    "max_depth": 3,
    "random_state": 42,
    "early_stopping_rounds": 10,
    "eval_metric": "logloss"
}

# 2. Huấn luyện (API mới tự động kích hoạt Early Stopping khi có eval_set)
xgb = XGBClassifier(**params)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

# 3. Thu thập Iterative metadata
mechanics = {
    "category": "iterative",
    "best_iteration": int(xgb.best_iteration),
    "best_score": float(xgb.best_score),
    "eval_history": xgb.evals_result()['validation_0']['logloss']
}

# 4. Đánh giá
y_pred = xgb.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average='weighted')
}

# 5. Lưu log
save_experiment_report(model_name, "iterative", params, metrics, mechanics)
joblib.dump(xgb, f"model/{model_name}/best_weights.pkl")
print(f"--- Kết quả {model_name} ---\n", classification_report(y_test, y_pred))

[Logger] Đã lưu báo cáo JSON cho XGBoost tại /model/XGBoost/experiment_log.json
--- Kết quả XGBoost ---
               precision    recall  f1-score   support

           0       1.00      0.98      0.99        52
           1       0.97      1.00      0.98        28

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



## 6. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Cấu hình
model_name = "RandomForest"
params = {"n_estimators": 100, "criterion": "gini", "random_state": 42}

# 2. Huấn luyện (Structural/Ensemble)
rf = RandomForestClassifier(**params)
rf.fit(X_train, y_train)

# 3. Thu thập metadata
mechanics = {
    "category": "non-iterative",
    "n_features_": int(rf.n_features_in_),
    "estimators_count": len(rf.estimators_)
}

# 4. Đánh giá
y_pred = rf.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average='weighted')
}

# 5. Lưu log
save_experiment_report(model_name, "non-iterative", params, metrics, mechanics)
joblib.dump(rf, f"model/{model_name}/best_weights.pkl")
print(f"--- Kết quả {model_name} ---\n", classification_report(y_test, y_pred))

[Logger] Đã lưu báo cáo JSON cho RandomForest tại /model/RandomForest/experiment_log.json
--- Kết quả RandomForest ---
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        52
           1       1.00      1.00      1.00        28

    accuracy                           1.00        80
   macro avg       1.00      1.00      1.00        80
weighted avg       1.00      1.00      1.00        80



## 7. Gradient Boosting Decision Tree - GBDT

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# 1. Cấu hình
model_name = "GBDT"
params = {"n_estimators": 100, "learning_rate": 0.1, "random_state": 42}

# 2. Huấn luyện (Iterative)
gbdt = GradientBoostingClassifier(**params)
gbdt.fit(X_train, y_train)

# 3. Thu thập metadata
mechanics = {
    "category": "iterative",
    "train_score": gbdt.train_score_.tolist(),
    "n_estimators_": int(gbdt.n_estimators_)
}

# 4. Đánh giá
y_pred = gbdt.predict(X_test)
metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "f1_weighted": f1_score(y_test, y_pred, average='weighted')
}

# 5. Lưu log
save_experiment_report(model_name, "iterative", params, metrics, mechanics)
joblib.dump(gbdt, f"model/{model_name}/best_weights.pkl")
print(f"--- Kết quả {model_name} ---\n", classification_report(y_test, y_pred))

[Logger] Đã lưu báo cáo JSON cho GBDT tại /model/GBDT/experiment_log.json
--- Kết quả GBDT ---
               precision    recall  f1-score   support

           0       1.00      0.98      0.99        52
           1       0.97      1.00      0.98        28

    accuracy                           0.99        80
   macro avg       0.98      0.99      0.99        80
weighted avg       0.99      0.99      0.99        80



# Tải xuống kết quả huấn luyện # Colab

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('models_export', 'zip', '/model')

files.download('models_export.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>